# CGM glucose forecasting: accuracy, confidence and tiny-device deployment

**Author:** Madiha Anjum

Part 1 (original work): forecasting continuous glucose monitoring (CGM) data with SARIMA, Prophet and XGBoost, using lag features.

Part 2 (new): how much can we **trust** each prediction, and what happens when the model is made small enough for a **microcontroller**.
- Prediction intervals with **conformal prediction** and **XGBoost quantile regression**
- A tiny neural network with **Monte Carlo dropout** confidence
- **8-bit (int8) quantization** with TensorFlow Lite: accuracy, model size, and whether the confidence interval still holds
- Export for **Edge Impulse** (latency/RAM/flash estimate on a target MCU) and a C header for TensorFlow Lite Micro

Upload `cgm5.csv` (columns `Time`, `Glucose` in mmol/L) to `/content/` before running.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
# from statsmodels.tsa.arima_model import ARIMA  # removed in new statsmodels, not used
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error, f1_score, r2_score, mean_absolute_percentage_error
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from statsmodels.graphics.tsaplots import plot_acf,plot_pacf


In [ ]:

df=pd.read_csv('/content/cgm5.csv')
# Load data from CSV file


# Ensure 'Time' column is in datetime format
df['Time'] = pd.to_datetime(df['Time'])

#df['Years'] = df['Time'].dt.year
df['Months'] = df['Time'].dt.month
#df['time'] = df['Time'].dt.time
df['Days_of_week'] = df['Time'].dt.dayofweek
df['Hours'] = df['Time'].dt.hour
#df['Minutes'] = df['Time'].dt.minute

# Feature engineering: Create lag features for XGBoost

for i in range(1, 4):  # Create lag features for the last 3 time points
  df[f'Glucose_Lag_{i}'] = df['Glucose'].shift(i)


# Feature engineering: Create ACF and PACF graphs features for Arima
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf
#plot_acf(df['Glucose'])

f=plt.figure()
ax1=f.add_subplot(121)
ax1.set_title('1st order differencing')
ax1.plot(df['Glucose'].diff())

ax2=f.add_subplot(122)
ax2.set_title('acf 1st order')
plot_acf(df['Glucose'].diff().dropna(),ax=ax2)
plt.show()

#f=plt.figure()
#ax1=f.add_subplot(121)
#ax1.set_title('2nd order differencing')
#ax1.plot(df['Glucose'].diff().diff())

f=plt.figure()
ax1=f.add_subplot(121)
ax1.set_title('8 order differencing')
ax1.plot(df['Glucose'].diff(8))

plot_pacf(df['Glucose'].diff(8).dropna(),lags=96)
ax2=f.add_subplot(122)
ax2.set_title('acf 2nd Order')
plot_acf(df['Glucose'].diff(8).dropna(),ax=ax2)
plt.show()

# adfuller
result= adfuller(df['Glucose'].dropna())
print('p-value original 1st:', result[1])
print('ADF Statistic 1st: %f' % result[0])

print('Critical Values 1st:')
for key, value in result[4].items():
	print('\t%s: %.3f' % (key, value))


#result= adfuller(df['Glucose'].diff().diff().dropna())
#print('p-value:', result[1])

# Drop NaN values resulting from lag features
df = df.dropna()

#remove outliers from our data set
# all readings are kept, including hypo- and hyperglycemic values (as in the paper)

# Split data into train and test sets
train_size = int(len(df) * 0.75)
train, test = df[0:train_size], df[train_size:]

#removing index from train test dataset
test=test.reset_index(drop=True)
train=train.reset_index(drop=True)

#test=test.set_index('Hours')
#train=train.set_index('Hours')

#code to plot train test dataset

data_test1=test
data_test1=data_test1.set_index('Months')
#data_test1=data_test1.set_index('Hours')
data_test1=data_test1.rename(columns={'Glucose': 'TEST SET'})

data_train1=train
data_train1=data_train1.set_index('Months')
#data_train1=data_train1.set_index('Hours')
data_train1=data_train1.rename(columns={'Glucose': 'TRAIN SET'})

data_test1=data_test1.drop(['Glucose_Lag_1','Glucose_Lag_2','Glucose_Lag_3','Time','Days_of_week','Hours'],axis=1)
data_train1=data_train1.drop(['Glucose_Lag_1','Glucose_Lag_2','Glucose_Lag_3','Time','Days_of_week','Hours'],axis=1)
color=['r', 'b', 'g', 'k', 'm']

data_test1.join(data_train1, how='outer').plot(figsize=(15,5), title='Glucose train test data', style='.')



# SARIMA model
sarima_model = SARIMAX(train['Glucose'], order=(8, 0,1), seasonal_order=(0,1,0,8))  # p=1, d=1 and q=2 (as two values are away from threshold )
sarima_result = sarima_model.fit()

# Prophet model
prophet_model = Prophet()
prophet_data = train.rename(columns={'Time': 'ds', 'Glucose': 'y'})
prophet_model.fit(prophet_data)

# XGBoost model
xgb_model = XGBRegressor(base_score=0.5, booster='gbtree', colsample_bylevel=1,
             colsample_bynode=1, colsample_bytree=1, gamma=0,
             importance_type='gain', interaction_constraints='',
             learning_rate=0.05, max_delta_step=0, max_depth=6,
             min_child_weight=1, monotone_constraints='()',
             n_estimators=500, n_jobs=0, num_parallel_tree=1, random_state=0,
             reg_alpha=0, reg_lambda=1, scale_pos_weight=1, subsample=1,
             tree_method='exact', validate_parameters=1, verbosity=None)

X_train, y_train = train.drop(['Time','Glucose'], axis=1), train['Glucose']
X_test, y_test = test.drop(['Time','Glucose'], axis=1), test['Glucose']
xgb_model.fit(X_train, y_train)

# Forecasting with ARIMA
sarima_forecast = sarima_result.predict(start=len(train), end=len(train) + len(test) - 1)  # SARIMAX already predicts levels

# Calculate ARIMA errors
sarima_r2 = r2_score(test['Glucose'], sarima_forecast)
sarima_mape = mean_absolute_percentage_error(test['Glucose'], sarima_forecast)
sarima_mae = mean_absolute_error(test['Glucose'], sarima_forecast)
sarima_mse = mean_squared_error(test['Glucose'], sarima_forecast)
sarima_rmse = np.sqrt(sarima_mse)

# Plot SARIMA results
plt.figure(figsize=(14, 6))
plt.plot(test['Time'], test['Glucose'], label='Historical Data')
plt.plot(test['Time'], sarima_forecast, label='SARIMA Forecast', linestyle='solid')
plt.title('SARIMA Forecasting\nr2: {:.2f}, MAPE: {:.2f}, MAE: {:.2f}, MSE: {:.2f}, RMSE: {:.2f}'.format(sarima_r2,sarima_mape,sarima_mae, sarima_mse, sarima_rmse))
plt.xlabel('Time')
plt.ylabel('Glucose Level')
plt.legend()
plt.show()

# Forecasting with Prophet
future_dates_prophet = pd.DataFrame({'ds': test['Time']})
prophet_forecast = prophet_model.predict(future_dates_prophet)
#r2_score, mean_absolute_percentage_error

# Calculate Prophet errors
prophet_r2 = r2_score(test['Glucose'], prophet_forecast['yhat'])
prophet_mape = mean_absolute_percentage_error(test['Glucose'], prophet_forecast['yhat'])
prophet_mae = mean_absolute_error(test['Glucose'], prophet_forecast['yhat'])
prophet_mse = mean_squared_error(test['Glucose'], prophet_forecast['yhat'])
prophet_rmse = np.sqrt(prophet_mse)

# Plot Prophet results
plt.figure(figsize=(14, 6))
prophet_model.plot(prophet_forecast, xlabel='Time', ylabel='Glucose Level')
plt.title('Prophet Forecasting\nr2: {:.2f}, MSE: {:.2f},MAPE: {:.2f}, MSE: {:.2f}, RMSE: {:.2f}'.format(prophet_r2, prophet_mape, prophet_mae, prophet_mse, prophet_rmse))
plt.show()

# Forecasting with XGBoost
xgb_forecast = xgb_model.predict(X_test)

# Calculate XGBoost errors
xgb_r2 = r2_score(y_test, xgb_forecast)
xgb_mape = mean_absolute_percentage_error(y_test, xgb_forecast)
xgb_mae = mean_absolute_error(y_test, xgb_forecast)
xgb_mse = mean_squared_error(y_test, xgb_forecast)
xgb_rmse = np.sqrt(xgb_mse)

# Plot XGBoost results
plt.figure(figsize=(14, 6))
plt.plot(test['Time'], test['Glucose'], label='Actual Data')
plt.plot(test['Time'], xgb_forecast, label='XGBoost Forecast', linestyle='solid')
plt.title('XGBoost Forecasting\nr2: {:.2f}, MAPE: {:.2f},MAE: {:.2f}, MSE: {:.2f}, RMSE: {:.2f}'.format(xgb_r2, xgb_mape, xgb_mae, xgb_mse, xgb_rmse))
plt.xlabel('Time')
plt.ylabel('Glucose Level')
plt.legend()
plt.show()

X_train.head()

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(['XGBoost','SARIMA', 'Prophet' ], [xgb_r2, sarima_r2, prophet_r2, ], marker='o', label='R^2')
plt.plot(['XGBoost','SARIMA', 'Prophet' ], [xgb_mape, sarima_mape, prophet_mape], marker='o', label='MAPE')
plt.plot(['XGBoost','SARIMA', 'Prophet' ], [xgb_mae, sarima_mae, prophet_mae], marker='o', label='MAE')
plt.plot(['XGBoost','SARIMA', 'Prophet' ], [xgb_mse, sarima_mse, prophet_mse], marker='o', label='MSE')
plt.plot(['XGBoost','SARIMA', 'Prophet' ], [xgb_rmse, sarima_rmse, prophet_rmse], marker='o', label='RMSE')

plt.title('Error Comparison -  XGBoost vs SARIMA vs Prophet ')
plt.xlabel('Model')
plt.ylabel('Error')
plt.legend()
plt.show()

In [ ]:
# Plot all forecasts in a single graph

#test_plot=test.reset_index(drop=True)
#test_plot=test_plot.set_index('Hours')

arima_forecast=sarima_forecast.reset_index(drop=True)
#arima_forecast_plot=arima_forecast_plot.set_index('Hours')

prophet_forecast=prophet_forecast.reset_index(drop=True)
#prophet_forecast_plot=prophet_forecast_plot.set_index('Hours')

#xgb_forecast=xgb_forecast.reset_index(drop=True)
#xgb_forecast_plot=xgb_forecast_plot.set_index('Hours')


#plt.figure(figsize=(15,5))
plt.plot(test['Glucose'],label='Observed Data', color='red', marker='o', linestyle='solid',markersize=3)
plt.plot(arima_forecast, label='ARIMA Forecast',marker='o', linestyle='solid',markersize=3)
plt.plot(prophet_forecast['yhat'], label='Prophet Forecast', marker='o',linestyle='solid',markersize=3)
plt.plot(xgb_forecast, label='XGBoost Forecast',marker='o', linestyle='solid',markersize=3)

plt.title('Forecast Comparison - ARIMA vs Prophet vs XGBoost')
plt.xlabel('Occurance')
plt.ylabel('Glucose Level')
plt.legend()
plt.show()


In [ ]:
# ... (Previous code)

# Calculate ARIMA errors
arima_mae = mean_absolute_error(test['Glucose'], arima_forecast)
arima_mse = mean_squared_error(test['Glucose'], arima_forecast)
arima_rmse = np.sqrt(arima_mse)

# Print ARIMA errors
print('ARIMA Forecasting Errors:')
print('MAE:', arima_mae)
print('MSE:', arima_mse)
print('RMSE:', arima_rmse)

# Plot ARIMA results
plt.figure(figsize=(14, 6))
plt.plot(df['Time'], df['Glucose'], label='Historical Data')
plt.plot(test['Time'], arima_forecast, label='ARIMA Forecast', linestyle='dashed')
plt.title('ARIMA Forecasting\nMAE: {:.2f}, MSE: {:.2f}, RMSE: {:.2f}'.format(arima_mae, arima_mse, arima_rmse))
plt.xlabel('Time')
plt.ylabel('Glucose Level')
plt.legend()
plt.show()

# ... (Repeat similar sections for Prophet and XGBoost)

# Print Prophet errors
print('\nProphet Forecasting Errors:')
print('MAE:', prophet_mae)
print('MSE:', prophet_mse)
print('RMSE:', prophet_rmse)

# Plot Prophet results
plt.figure(figsize=(14, 6))
prophet_model.plot(prophet_forecast, xlabel='Time', ylabel='Glucose Level')
plt.title('Prophet Forecasting\nMAE: {:.2f}, MSE: {:.2f}, RMSE: {:.2f}'.format(prophet_mae, prophet_mse, prophet_rmse))
plt.show()

# ... (Repeat similar sections for XGBoost)

# Print XGBoost errors
print('\nXGBoost Forecasting Errors:')
print('MAE:', xgb_mae)
print('MSE:', xgb_mse)
print('RMSE:', xgb_rmse)

# Plot XGBoost results
plt.figure(figsize=(14, 6))
plt.plot(test['Time'], test['Glucose'], label='Actual Data')
plt.plot(test['Time'], xgb_forecast, label='XGBoost Forecast', linestyle='dashed')
plt.title('XGBoost Forecasting\nMAE: {:.2f}, MSE: {:.2f}, RMSE: {:.2f}'.format(xgb_mae, xgb_mse, xgb_rmse))
plt.xlabel('Time')
plt.ylabel('Glucose Level')
plt.legend()
plt.show()


---
# Part 2: Confidence and tiny-device deployment

A forecast alone is not enough for a patient device: the system should also say **how sure it is**.
Below, every prediction gets a 90% interval, and we check the **coverage** (how often the true glucose value falls inside the interval; target 90%).

In [ ]:
import time, os
import tensorflow as tf

ALPHA = 0.10          # 90% intervals
FEATURES = list(X_train.columns)
print('Features:', FEATURES)

def coverage(y, lo, hi):
    y = np.asarray(y); return float(np.mean((y >= lo) & (y <= hi)))

# Time-ordered split of the training data: last 20% is kept for calibration
n_cal = int(len(X_train) * 0.2)
X_fit, y_fit = X_train.iloc[:-n_cal], y_train.iloc[:-n_cal]
X_cal, y_cal = X_train.iloc[-n_cal:], y_train.iloc[-n_cal:]
print('fit:', len(X_fit), ' calibration:', len(X_cal), ' test:', len(X_test))

## 2.1 Conformal prediction for XGBoost

Train the same XGBoost on the first 80% of the training data, measure its errors on the calibration part,
and use the 90th percentile of those errors as the interval half-width. This gives a coverage guarantee
without changing the model, and on a device it costs only one addition and one subtraction.

In [ ]:
xgb_c = XGBRegressor(learning_rate=0.05, max_depth=6, n_estimators=500, random_state=0, tree_method='exact')
xgb_c.fit(X_fit, y_fit)

cal_err = np.abs(y_cal.values - xgb_c.predict(X_cal))
q_level = min(1.0, np.ceil((len(cal_err) + 1) * (1 - ALPHA)) / len(cal_err))
q_xgb = float(np.quantile(cal_err, q_level, method='higher'))

pred_c = xgb_c.predict(X_test)
lo_c, hi_c = pred_c - q_xgb, pred_c + q_xgb
print(f'Conformal half-width: +/- {q_xgb:.2f} mmol/L')
print(f'Test MAE: {mean_absolute_error(y_test, pred_c):.3f}')
print(f'Coverage: {coverage(y_test, lo_c, hi_c):.1%} (target {1-ALPHA:.0%})')

## 2.2 XGBoost quantile regression (interval width changes with the situation)

Conformal intervals have the same width everywhere. Quantile regression learns the 5th and 95th percentiles directly,
so the interval can be wider after meals and narrower at night. (Needs XGBoost 2.0 or newer; Colab already has it.)

In [ ]:
xgb_q = XGBRegressor(objective='reg:quantileerror', quantile_alpha=np.array([ALPHA/2, 0.5, 1-ALPHA/2]),
                     learning_rate=0.05, max_depth=6, n_estimators=500, random_state=0, tree_method='hist')
xgb_q.fit(X_train, y_train)
q_pred = xgb_q.predict(X_test)
lo_q, med_q, hi_q = q_pred[:, 0], q_pred[:, 1], q_pred[:, 2]
print(f'Median MAE: {mean_absolute_error(y_test, med_q):.3f}')
print(f'Coverage: {coverage(y_test, lo_q, hi_q):.1%}   mean width: {np.mean(hi_q-lo_q):.2f} mmol/L')

plt.figure(figsize=(14, 5))
k = slice(0, 300)
plt.plot(test['Time'][k], y_test.values[k], 'r.', markersize=4, label='Actual')
plt.plot(test['Time'][k], pred_c[k], label='XGBoost forecast')
plt.fill_between(test['Time'][k], lo_c[k], hi_c[k], alpha=0.2, label='Conformal 90% interval')
plt.fill_between(test['Time'][k], lo_q[k], hi_q[k], alpha=0.2, label='Quantile 90% interval')
plt.title('XGBoost forecast with 90% prediction intervals'); plt.ylabel('Glucose (mmol/L)'); plt.legend(); plt.show()

## 2.3 A tiny neural network for a microcontroller

XGBoost with 500 trees is too large for most microcontrollers. A small neural network (6 inputs, two hidden layers of 16)
fits in a few KB. Dropout layers let us estimate confidence with **Monte Carlo dropout**: run the model 50 times with dropout on
and use the spread of the outputs.

In [ ]:
tf.keras.utils.set_random_seed(0)
mu, sd = X_fit.mean(), X_fit.std().replace(0, 1)
norm = lambda X: ((X - mu) / sd).values.astype('float32')

mlp = tf.keras.Sequential([
    tf.keras.Input(shape=(len(FEATURES),)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dropout(0.1),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dropout(0.1),
    tf.keras.layers.Dense(1)])
mlp.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mae')
mlp.fit(norm(X_fit), y_fit.values, validation_data=(norm(X_cal), y_cal.values), epochs=200, batch_size=64, verbose=0,
        callbacks=[tf.keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True)])

pred_f = mlp.predict(norm(X_test), verbose=0).ravel()
print(f'MLP float32 MAE: {mean_absolute_error(y_test, pred_f):.3f}   parameters: {mlp.count_params()}')

# Monte Carlo dropout: 50 passes with dropout switched on
mc = np.stack([mlp(norm(X_test), training=True).numpy().ravel() for _ in range(50)])
mc_mean, mc_std = mc.mean(0), mc.std(0)
z = 1.645
print(f'MC dropout coverage: {coverage(y_test, mc_mean - z*mc_std, mc_mean + z*mc_std):.1%}   mean width: {np.mean(2*z*mc_std):.2f}')

MC dropout is often **over-confident** (coverage below 90%), and it needs 50 runs per prediction, which is too slow for a small device.
Conformal calibration fixes both: compute the half-width once on the PC, store it as one number on the device.

In [ ]:
cal_err_f = np.abs(y_cal.values - mlp.predict(norm(X_cal), verbose=0).ravel())
q_f = float(np.quantile(cal_err_f, min(1.0, np.ceil((len(cal_err_f)+1)*(1-ALPHA))/len(cal_err_f)), method='higher'))
print(f'MLP float32 conformal half-width: +/- {q_f:.2f}   coverage: {coverage(y_test, pred_f - q_f, pred_f + q_f):.1%}')

## 2.4 8-bit (int8) quantization with TensorFlow Lite

Full-integer quantization makes the model about 4x smaller and runs on MCUs without a floating-point unit.
The key question for a dependable device: **does the confidence interval calibrated on the float model still hold after quantization?**

In [ ]:
conv = tf.lite.TFLiteConverter.from_keras_model(mlp)
tfl_f32 = conv.convert()

def rep_data():
    for row in norm(X_fit)[:500]:
        yield [row.reshape(1, -1)]

conv = tf.lite.TFLiteConverter.from_keras_model(mlp)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = rep_data
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type = tf.int8
conv.inference_output_type = tf.int8
tfl_i8 = conv.convert()

def run_tflite(model_bytes, X):
    it = tf.lite.Interpreter(model_content=model_bytes); it.allocate_tensors()
    inp, out = it.get_input_details()[0], it.get_output_details()[0]
    res = []
    for row in X:
        x = row.reshape(1, -1)
        if inp['dtype'] == np.int8:
            s, zp = inp['quantization']; x = np.clip(np.round(x / s + zp), -128, 127).astype(np.int8)
        it.set_tensor(inp['index'], x); it.invoke()
        y = it.get_tensor(out['index']).astype('float32')
        if out['dtype'] == np.int8:
            s, zp = out['quantization']; y = (y - zp) * s
        res.append(y.ravel()[0])
    return np.array(res)

pred_i8 = run_tflite(tfl_i8, norm(X_test))
cal_i8 = run_tflite(tfl_i8, norm(X_cal))
q_i8 = float(np.quantile(np.abs(y_cal.values - cal_i8), min(1.0, np.ceil((len(cal_i8)+1)*(1-ALPHA))/len(cal_i8)), method='higher'))

print(f'int8 MAE: {mean_absolute_error(y_test, pred_i8):.3f}')
print(f'int8 with FLOAT-calibrated interval (+/-{q_f:.2f}): coverage {coverage(y_test, pred_i8 - q_f, pred_i8 + q_f):.1%}')
print(f'int8 with re-calibrated interval   (+/-{q_i8:.2f}): coverage {coverage(y_test, pred_i8 - q_i8, pred_i8 + q_i8):.1%}')

## 2.5 Summary table

In [ ]:
xgb_path = 'xgb_model.json'; xgb_c.save_model(xgb_path)
t0 = time.perf_counter(); run_tflite(tfl_i8, norm(X_test)[:200]); t_i8 = (time.perf_counter() - t0) / 200 * 1e6

summary = pd.DataFrame([
    ['XGBoost (500 trees)', mean_absolute_error(y_test, pred_c), np.sqrt(mean_squared_error(y_test, pred_c)),
     os.path.getsize(xgb_path)/1024, f'+/-{q_xgb:.2f}', coverage(y_test, lo_c, hi_c)],
    ['MLP float32', mean_absolute_error(y_test, pred_f), np.sqrt(mean_squared_error(y_test, pred_f)),
     len(tfl_f32)/1024, f'+/-{q_f:.2f}', coverage(y_test, pred_f - q_f, pred_f + q_f)],
    ['MLP int8', mean_absolute_error(y_test, pred_i8), np.sqrt(mean_squared_error(y_test, pred_i8)),
     len(tfl_i8)/1024, f'+/-{q_i8:.2f}', coverage(y_test, pred_i8 - q_i8, pred_i8 + q_i8)],
], columns=['Model', 'MAE', 'RMSE', 'Size (KB)', '90% interval', 'Coverage'])
print(summary.round(3).to_string(index=False))
print(f'\n(int8 inference on this Colab CPU: {t_i8:.0f} us per prediction; use Edge Impulse for MCU numbers)')

## 2.6 Export for Edge Impulse and for a microcontroller

1. `cgm_mlp_int8.tflite`: upload to **Edge Impulse** (free account, *Bring your own model*), choose a target such as
   Arduino Nano 33 BLE Sense (Cortex-M4F) or ESP32, and it reports estimated **latency, RAM and flash**.
2. `cgm_model_data.h`: the same model as a C array for **TensorFlow Lite Micro**, plus the normalization values and the
   conformal half-width, so the device can output *glucose forecast +/- interval*.

In [ ]:
open('cgm_mlp_int8.tflite', 'wb').write(tfl_i8)

lines = [f'// CGM forecaster, int8 TFLite model ({len(tfl_i8)} bytes)',
         f'// Inputs (in order): {FEATURES}',
         'const float kFeatureMean[] = {' + ', '.join(f'{v:.6f}f' for v in mu.values) + '};',
         'const float kFeatureStd[]  = {' + ', '.join(f'{v:.6f}f' for v in sd.values) + '};',
         f'const float kIntervalHalfWidth = {q_i8:.4f}f;  // 90% conformal interval, mmol/L',
         f'const unsigned int kModelLen = {len(tfl_i8)};',
         'alignas(16) const unsigned char kModel[] = {']
for i in range(0, len(tfl_i8), 12):
    lines.append('  ' + ', '.join(f'0x{b:02x}' for b in tfl_i8[i:i+12]) + ',')
lines.append('};')
open('cgm_model_data.h', 'w').write('\n'.join(lines) + '\n')
print('Saved cgm_mlp_int8.tflite and cgm_model_data.h')

try:
    from google.colab import files
    files.download('cgm_mlp_int8.tflite'); files.download('cgm_model_data.h')
except ImportError:
    pass